# 📷 Smartphone → DSLR Enhancement using CycleGAN

**Paper:** Zhu et al., *Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks*, ICCV 2017

**Dataset:** DPED — Smartphone vs DSLR flower photos (unpaired)

---
### Steps in this notebook:
1. Setup GPU + clone repo
2. Install dependencies
3. Download DPED dataset
4. Train CycleGAN
5. Visualize results
6. Download trained weights

## Cell 1 — Check GPU

In [2]:
import torch

print('=' * 45)
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found — Runtime > Change runtime type > T4 GPU')
print('=' * 45)

GPU : Tesla T4
VRAM: 15.6 GB


## Cell 2 — Clone GitHub Repo

In [4]:
import os

REPO_URL  = 'https://github.com/sidrahrahim5-cmyk/smartphone2dslr-cyclegan.git'
REPO_NAME = 'smartphone2dslr-cyclegan'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone {REPO_URL}')
    print('Repo cloned!')
else:
    os.system(f'cd {REPO_NAME} && git pull')
    print('Repo updated!')

os.chdir(REPO_NAME)
print(f'Working directory: {os.getcwd()}')

Repo cloned!
Working directory: /content/smartphone2dslr-cyclegan/smartphone2dslr-cyclegan


## Cell 3 — Install Dependencies

In [6]:
os.system('pip install -q -r requirements.txt')
print('Dependencies installed!')

Dependencies installed!


## Cell 4 — Download DPED Dataset

DPED dataset mein iPhone aur Canon DSLR flower photos hain.
Hum unpaired subset use karenge CycleGAN ke liye.

In [7]:
import os
import zipfile
import gdown

os.makedirs('dataset/trainA', exist_ok=True)
os.makedirs('dataset/trainB', exist_ok=True)
os.makedirs('dataset/testA',  exist_ok=True)
os.makedirs('dataset/testB',  exist_ok=True)

# DPED iphone flowers (Domain A — Smartphone)
print('Downloading smartphone photos (Domain A)...')
url_A = 'https://drive.google.com/uc?id=1V3yCdi1DoEI7APFZ9R3bHMmj7Xh5NQXU'
gdown.download(url_A, 'iphone_flowers.zip', quiet=False)

# DPED Canon DSLR flowers (Domain B — DSLR)
print('Downloading DSLR photos (Domain B)...')
url_B = 'https://drive.google.com/uc?id=1V3yCdi1DoEI7APFZ9R3bHMmj7Xh5NQXU'
gdown.download(url_B, 'canon_flowers.zip', quiet=False)

print('Dataset download complete!')

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1V3yCdi1DoEI7APFZ9R3bHMmj7Xh5NQXU

but Gdown can't. Please check connections and permissions.

## Cell 4B — Alternative: Use CycleGAN Official Dataset

Agar DPED download na ho toh official CycleGAN dataset use karo.
Ye guaranteed kaam karta hai!

In [9]:
import os
import json
import zipfile

# ── Fill in your Kaggle credentials ──────────────────────────
KAGGLE_USERNAME = "sidrahrahim"    # Replace with your username
KAGGLE_KEY      = "KGAT_88c38e0a9bf6280a43db5e01e0a2bcd2"       # Replace with your API key
# ─────────────────────────────────────────────────────────────

# Step 1: Configure Kaggle credentials
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

credentials = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(credentials, f)

os.system("chmod 600 ~/.kaggle/kaggle.json")
print("Kaggle credentials configured!")

# Step 2: Install Kaggle CLI
os.system("pip install -q kaggle")

# Step 3: Download dataset
print("Downloading iphone2dslr flower dataset...")
os.system("kaggle datasets download -d balraj98/iphone2dslr-flower-dataset")

# Step 4: Extract dataset
print("Extracting...")
with zipfile.ZipFile("iphone2dslr-flower-dataset.zip", "r") as zf:
    zf.extractall(".")

# Step 5: Organize into dataset folders
for split in ["trainA", "trainB", "testA", "testB"]:
    os.makedirs(f"dataset/{split}", exist_ok=True)

os.system("cp -r iphone2dslr_flower/trainA/. dataset/trainA/")
os.system("cp -r iphone2dslr_flower/trainB/. dataset/trainB/")
os.system("cp -r iphone2dslr_flower/testA/.  dataset/testA/")
os.system("cp -r iphone2dslr_flower/testB/.  dataset/testB/")

# Step 6: Verify
print("\nDataset summary:")
for split in ["trainA", "trainB", "testA", "testB"]:
    path = f"dataset/{split}"
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f"  {path}: {n} images")
    else:
        print(f"  {path}: NOT FOUND")

print("\nDataset ready for training!")

URLError: <urlopen error [Errno -2] Name or service not known>

## Cell 5 — Verify Dataset + Show Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_samples(folder, title, n=4):
    files = [f for f in os.listdir(folder)
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    sample = random.sample(files, min(n, len(files)))

    fig, axes = plt.subplots(1, n, figsize=(16, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    for ax, fname in zip(axes, sample):
        img = Image.open(os.path.join(folder, fname)).convert('RGB')
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples('dataset/trainA', 'Domain A — Smartphone Photos')
show_samples('dataset/trainB', 'Domain B — DSLR Photos')

## Cell 6 — Configure Training

In [ ]:
# Override config for Colab
# Edit these values if needed

import sys
sys.path.insert(0, '.')

from config import Config

# Colab-specific overrides
Config.DEVICE       = 'cuda'
Config.EPOCHS       = 50      # Start with 50 for Colab (full=200)
Config.DECAY_EPOCH  = 25
Config.BATCH_SIZE   = 1
Config.IMG_SIZE     = 256
Config.LOG_EVERY    = 200     # Save sample every 200 batches

Config.make_dirs()

print('Training Configuration:')
print(f'  Device        : {Config.DEVICE}')
print(f'  Epochs        : {Config.EPOCHS}')
print(f'  Image size    : {Config.IMG_SIZE}x{Config.IMG_SIZE}')
print(f'  Lambda cycle  : {Config.LAMBDA_CYCLE}')
print(f'  Lambda identity: {Config.LAMBDA_IDENTITY}')
print(f'  Learning rate : {Config.LR}')

## Cell 7 — Start Training

In [ ]:
# This runs the full training loop from train.py
from train import train
train()

## Cell 8 — Visualize Training Results

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

# Show latest saved sample grids
result_files = sorted(glob.glob('assets/results/*.png'))

if not result_files:
    print('No results yet — run training first!')
else:
    # Show first, middle and last result
    indices = [0, len(result_files)//2, -1]
    labels  = ['Early Training', 'Mid Training', 'Final Result']

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax, idx, label in zip(axes, indices, labels):
        img = Image.open(result_files[idx])
        ax.imshow(img)
        ax.set_title(label, fontsize=13, fontweight='bold')
        ax.axis('off')

    plt.suptitle('CycleGAN Training Progress', fontsize=15)
    plt.tight_layout()
    plt.show()
    print(f'Total result files saved: {len(result_files)}')

## Cell 9 — Test on a Custom Image

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

from config      import Config
from src.models  import Generator
from src.utils   import tensor_to_image

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained generator
G = Generator(config.IMG_CHANNELS, config.N_RESIDUAL_BLOCKS).to(device)
ckpt = torch.load('checkpoints/latest.pth', map_location=device)
G.load_state_dict(ckpt['G'])
G.eval()
print('Generator loaded!')

# Test on a random image from testA
test_images = os.listdir('dataset/testA')
test_path   = os.path.join('dataset/testA', test_images[0])

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(256),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

img    = Image.open(test_path).convert('RGB')
tensor = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    enhanced = G(tensor)

# Show comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img.resize((256, 256)))
axes[0].set_title('Input: Smartphone Photo', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(tensor_to_image(enhanced[0]))
axes[1].set_title('Output: DSLR Quality', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Smartphone → DSLR Enhancement', fontsize=15)
plt.tight_layout()
plt.savefig('assets/results/test_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Test complete!')

## Cell 10 — Download Trained Weights

In [ ]:
from google.colab import files
import os

# Download latest checkpoint
checkpoint = 'checkpoints/latest.pth'

if os.path.exists(checkpoint):
    files.download(checkpoint)
    print('Checkpoint downloaded!')
    print('Save it in your local checkpoints/ folder')
else:
    print('No checkpoint found — run training first!')

# Also download test result
result = 'assets/results/test_result.png'
if os.path.exists(result):
    files.download(result)
    print('Test result image downloaded!')